In [1]:
import pandas as pd
import numpy as np
import chromadb
from chromadb.utils import embedding_functions
import time

# 1. Cargar el dataset procesado
df = pd.read_csv("../data/processed/reviews_en_clean.csv")

# Asegurar formato de horas jugadas y recommended
if 'author.playtime_forever' in df.columns:
    df['hours_played'] = (df['author.playtime_forever'] / 60.0).round(1)
elif 'playtime_forever' in df.columns:
    df['hours_played'] = (df['playtime_forever'] / 60.0).round(1)
else:
    df['hours_played'] = 0.0

df['recommended_str'] = df['recommended'].map({True: 'Recomendado', False: 'No Recomendado', 1: 'Recomendado', 0: 'No Recomendado'})

print(f"Total de registros disponibles: {len(df):,}")

Total de registros disponibles: 166,191


In [2]:
# 1. Configurar cliente persistente de ChromaDB
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")

# 2. Configurar función de embedding estándar de Sentence Transformers
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 3. Crear u obtener la colección
# Usamos métrica 'cosine' para medir la distancia angular semántica
collection = chroma_client.get_or_create_collection(
    name="steam_reviews",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

print(f"Colección lista. Registros actuales indexados: {collection.count()}")

c:\My proyects\steam_review_analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2031.39it/s]


Colección lista. Registros actuales indexados: 0


In [3]:
# Seleccionamos una muestra balanceada de 10,000 reseñas para indexar y probar
# Es posible ajustar esa cantidad o incluso filtrar por juegos específicos si se desea
sample_df = df.sample(n=min(10000, len(df)), random_state=42).reset_index(drop=True)

batch_size = 500   # Tamaño de lote para inserción en ChromaDB, puede ajustarse según memoria y rendimiento
total_samples = len(sample_df)

print(f"Iniciando indexación de {total_samples:,} reseñas...")
start_time = time.time()

for i in range(0, total_samples, batch_size):
    batch = sample_df.iloc[i:i + batch_size]
    
    # Preparar IDs únicos, textos y metadatos
    ids = [str(row['review_id']) if 'review_id' in row and pd.notna(row['review_id']) else f"rev_{i+idx}" for idx, (_, row) in enumerate(batch.iterrows())]
    documents = batch['review'].astype(str).tolist()
    
    metadatas = []
    for _, row in batch.iterrows():
        metadatas.append({
            "app_name": str(row.get('app_name', 'Unknown')),
            "recommended": str(row.get('recommended_str', 'Unknown')),
            "hours_played": float(row.get('hours_played', 0.0))
        })
        
    # Inserción en ChromaDB
    collection.upsert(
        ids=ids,
        documents=documents,
        metadatas=metadatas
    )
    print(f"Indexados: {min(i + batch_size, total_samples)} / {total_samples}")

elapsed = time.time() - start_time
print(f"Indexación completada con éxito en {elapsed:.2f} segundos.")

Iniciando indexación de 10,000 reseñas...
Indexados: 500 / 10000
Indexados: 1000 / 10000
Indexados: 1500 / 10000
Indexados: 2000 / 10000
Indexados: 2500 / 10000
Indexados: 3000 / 10000
Indexados: 3500 / 10000
Indexados: 4000 / 10000
Indexados: 4500 / 10000
Indexados: 5000 / 10000
Indexados: 5500 / 10000
Indexados: 6000 / 10000
Indexados: 6500 / 10000
Indexados: 7000 / 10000
Indexados: 7500 / 10000
Indexados: 8000 / 10000
Indexados: 8500 / 10000
Indexados: 9000 / 10000
Indexados: 9500 / 10000
Indexados: 10000 / 10000
Indexación completada con éxito en 102.11 segundos.


In [6]:
query = "terrible performance stuttering and low fps in combat"

results = collection.query(
    query_texts=[query],
    n_results=3,
    # Opcional: filtrar solo quejas técnicas o de un juego específico
    where={"recommended": "No Recomendado"}
)

print(f"\nConsulta: '{query}'\n")
for idx, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
    print(f"--- Resultado {idx + 1} (Distancia Coseno: {dist:.4f}) ---")
    print(f"Juego: {meta['app_name']} | Estado: {meta['recommended']} | Horas: {meta['hours_played']} hs")
    print(f"Texto: {doc[:500]}...\n")


Consulta: 'terrible performance stuttering and low fps in combat'

--- Resultado 1 (Distancia Coseno: 0.4587) ---
Juego: The Witcher 3: Wild Hunt | Estado: No Recomendado | Horas: 58.2 hs
Texto: I really love the game. Immersive and gripping story, really well written dialouges and characters and a decent but somewhat dull combat system. What really dissapoints me is the performance. I'm well above the recommended hardware settings with the GPU and my CPU is also above the minimum requirements. I tried to play it with the recommended settings, but couldn't really enjoy it. Even after tuning down most of the graphical options the game still runs badly with an average fps arround 40 and c...

--- Resultado 2 (Distancia Coseno: 0.4875) ---
Juego: The Witcher 3: Wild Hunt | Estado: No Recomendado | Horas: 6.6 hs
Texto: I bought this game because of the great reviews this series has, but frankly I'm disappointed. The combat is clunky and repetitive, the characters are unlikable and boring.

In [7]:
# Función  para recuperar reseñas semánticamente afines y formatearlas como contexto para un LLM
def retrieve_reviews_context(query: str, n_results: int = 3, filter_negative_only: bool = True) -> str:
    """Recupera reseñas semánticamente afines y las formatea como bloque de contexto para un LLM."""
    where_clause = {"recommended": "No Recomendado"} if filter_negative_only else None
    
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        where=where_clause
    )
    
    formatted_context = []
    for idx, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        entry = (
            f"[Reseña {idx + 1} | Juego: {meta['app_name']} | "
            f"Horas: {meta['hours_played']} hs | Voto: {meta['recommended']}]\n"
            f"{doc.strip()}"
        )
        formatted_context.append(entry)
        
    return "\n\n---\n\n".join(formatted_context)

# Prueba rápida del Retriever empaquetado
contexto_ejemplo = retrieve_reviews_context("constant crashes after latest update", n_results=2)
print("Contexto formateado para inyectar al LLM:\n")
print(contexto_ejemplo)

Contexto formateado para inyectar al LLM:

[Reseña 1 | Juego: The Witcher 3: Wild Hunt | Horas: 96.4 hs | Voto: No Recomendado]
I've been playing this game for about 2 hours... It just keep crashing and crashing... due to the glitches of the game. I own a GTX 980, it actually runs very smooth but suddenly out of nowhere, it crashes or freezes. Just need an explanation please. I really want to keep playing this game.

---

[Reseña 2 | Juego: The Witcher 3: Wild Hunt | Horas: 51.6 hs | Voto: No Recomendado]
there is a bug and i cannot save my game. dont know why. this sucks


## Metodología y decisiones técnicas de la fase 5: Indexación Vectorial y RAG

### 1. Modelo de embeddings seleccionado
* **Modelo:** `all-MiniLM-L6-v2` (arquitectura basada en Transformers de 6 capas y 384 dimensiones).
* **Justificación técnica:** Ofrece una relación costo-rendimiento óptima para inferencia en CPU local, con baja latencia y alta precisión semántica en similitud angular.

### 2. Base de datos vectorial: ChromaDB
* **Mecanismo de Persistencia:** Almacenamiento local indexado mediante algoritmo HNSW (*Hierarchical Navigable Small World*) utilizando distancia de coseno (`hnsw:space: "cosine"`).
* **Ingesta por Lotes (Batch Ingestion):** Carga segmentada en bloques de 500 registros para optimizar el uso de RAM y evitar cuellos de botella de I/O.

### 3. Filtrado híbrido: Semántica + Metadatos
* **Problema detectado:** La similitud vectorial pura proyecta proximidad temática sin distinguir necesariamente la polaridad del usuario (ej. reseñas positivas y negativas que mencionan problemas de "stuttering").
* **Solución arquitectural:** Implementación de filtrado booleano previo por metadatos (`where={"recommended": "No Recomendado"}`) previo a la búsqueda por similitud, garantizando la recuperación exclusiva de quejas técnicas relevantes para el triage de desarrollo.